In [ ]:
from pathlib import Path
import numpy as np
import pickle
import torch
import json
from glob import glob
from tqdm import tqdm

In [ ]:
# Loading nuclr embeddings
ckpt_dir = Path("../embs/<run-id!!!! MUST FILL>")  # transductive/inductive depends on what the model was trained on


ckpt_paths = sorted(ckpt_dir.iterdir(), key=lambda x: int(x.stem.split("_")[-1]))
print(f"Found {len(ckpt_paths)} checkpoints")

ids = None
embs = []  # List of embeddings from different checkpoints
for path in tqdm(ckpt_paths):
    emb_data = torch.load(path, weights_only=False, map_location="cpu")

    embs.append(emb_data["embs"].float().numpy())
    if ids is None:
        ids = emb_data["ids"]
    else:
        assert (ids == emb_data["ids"]).all()

## Transductive

In [ ]:
# Load split

splits_file = Path("../splits/steinmetz_within.pkl")
with open(splits_file, "rb") as f:
    splits_data = pickle.load(f)

label_map = splits_data["label_map"]
split = splits_data["splits"]

In [ ]:
# Remove UUIDs not known to splits

def to_mask(check_ids, base_ids):
    """
    Returns a bool array indicating if elements of `check_ids` are 
    present in `base_ids`. 
    Output shape: (len(check_ids),)
    """
    base_ids_set = set(base_ids)
    return np.array([x in base_ids_set for x in check_ids])

# Subset units based on what labels are available
valid_id_mask = to_mask(ids, label_map.index.values)
print(f"Valid UUID stats: {valid_id_mask.mean()=}, {(~valid_id_mask).sum()=}")

valid_ids = ids[valid_id_mask]
#valid_labels = data["curated_cluster_cosmos"][valid_id_mask]
valid_labels = label_map.loc[valid_ids].to_numpy()
valid_embs = []
for emb in tqdm(embs):
    valid_embs.append(emb[valid_id_mask])

In [ ]:
from cuml.linear_model import LogisticRegression
from cuml.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
from tqdm import tqdm
import numpy as np
import ray

seed = 42

def balanced_resampling(X, y):
    resample = RandomOverSampler(random_state=seed)
    resample_idx, _ = resample.fit_resample(np.arange(len(y)).reshape(-1, 1), y)
    resample_idx = resample_idx.ravel()
    return X[resample_idx], y[resample_idx]

def logreg_scores(X, y, train_mask, val_mask, test_mask):
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    X_train, y_train = balanced_resampling(X_train, y_train)

    scaler = StandardScaler()
    clf = LogisticRegression(
            max_iter=1000,
            tol=1e-5,
            #class_weight="balanced",
            #fit_intercept=True,
            C=1.0,
            #solver="lbfgs",
            #solver="saga",
            #n_jobs=32,
            verbose=0,
            #random_state=42,
            #multi_class='multinomial',
    )
    clf.fit(scaler.fit_transform(X_train), y_train)
    pred_val = clf.predict(scaler.transform(X_val))
    pred_test = clf.predict(scaler.transform(X_test))

    return {
        "val_f1": f1_score(y_val, pred_val, average='macro'),
        "val_bacc": balanced_accuracy_score(y_val, pred_val),
        "val_report": classification_report(y_val, pred_val),
        "test_f1": f1_score(y_test, pred_test, average='macro'),
        "test_bacc": balanced_accuracy_score(y_test, pred_test),
        "test_report": classification_report(y_test, pred_test),
    }

@ray.remote(num_cpus=1, num_gpus=0.1)
def ray_logreg_scores(X, y, train_mask, val_mask, test_mask):
    return logreg_scores(X, y, train_mask, val_mask, test_mask)

ray.init(address="local", num_cpus=32, num_gpus=1, ignore_reinit_error=True)


train_ids = split["train"]
train_mask = to_mask(valid_ids, train_ids)
val_mask = to_mask(valid_ids, split["val"])
test_mask = to_mask(valid_ids, split["test"])

y = valid_labels
futures = []
for i, X in tqdm(enumerate(valid_embs), total=len(valid_embs)):
    _future = ray_logreg_scores.remote(X, y, train_mask, val_mask, test_mask)
    futures.append(_future)

scores_list = []
for i, future in enumerate(futures):
    scores = ray.get(future)
    scores["index"] = i
    scores_list.append(scores)
    print(
        f"Index {i} | "
        f"Validation - bacc: {scores['val_bacc']:.4f}, F1: {scores['val_f1']:.4f} | "
        f"Test - bacc: {scores['test_bacc']:.4f}, F1: {scores['test_f1']:.4f}"
    )

best_scores = sorted(scores_list, key=lambda x: x["val_f1"], reverse=True)[0]
print(f"Best index: {best_scores['index']}")
print(f"Validation bal. acc.: {best_scores['val_bacc']:.4f}")
print(f"Validation F1: {best_scores['val_f1']:.4f}")
print(f"Test bal. acc.: {best_scores['test_bacc']:.4f}") 
print(f"Test F1: {best_scores['test_f1']:.4f}")
print(f"Best checkpoint: {ckpt_paths[best_scores['index']].name}")

## Transductive zero-shot

In [ ]:
# Load splits
splits_file = Path("../splits/steinmetz_subjectwise.pkl")
with open(splits_file, "rb") as f:
    splits_data = pickle.load(f)

label_map = splits_data["label_map"]
splits = splits_data["splits"]

In [ ]:
# Remove UUIDs not known to splits
def to_mask(check_ids, base_ids):
    """
    Returns a bool array indicating if elements of `check_ids` are 
    present in `base_ids`. 
    Output shape: (len(check_ids),)
    """
    base_ids_set = set(base_ids)
    return np.array([x in base_ids_set for x in check_ids])

# Subset units based on what labels are available
valid_id_mask = to_mask(ids, label_map.index.values)
print(f"Valid UUID stats: {valid_id_mask.mean()=}, {(~valid_id_mask).sum()=}")

valid_ids = ids[valid_id_mask]
valid_labels = label_map.loc[valid_ids].to_numpy().ravel()
valid_embs = []
for emb in tqdm(embs):
    valid_embs.append(emb[valid_id_mask])

In [ ]:
from cuml.linear_model import LogisticRegression
from cuml.preprocessing import StandardScaler
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import ray

seed = 42

def balanced_resampling(X, y):
    resample = RandomOverSampler(random_state=seed)
    resample_idx, _ = resample.fit_resample(np.arange(len(y)).reshape(-1, 1), y)
    resample_idx = resample_idx.ravel()
    return X[resample_idx], y[resample_idx]
    
def logreg_scores(X, y, train_mask, val_mask):
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    
    X_train, y_train = balanced_resampling(X_train, y_train)
    scaler = StandardScaler()
    clf = LogisticRegression(
            max_iter=1000,
            tol=1e-5,
            C=1.0,
            verbose=0,
    )
    clf.fit(scaler.fit_transform(X_train), y_train)
    pred_val = clf.predict(scaler.transform(X_val))

    return {
        "pred": pred_val,
        "true": y_val,
    }

@ray.remote(num_cpus=1, num_gpus=0.1)
def ray_logreg_scores(X, y, train_mask, val_mask):
    return logreg_scores(X, y, train_mask, val_mask)

ray.init(address="local", num_cpus=16, num_gpus=1, ignore_reinit_error=True)

num_ckpt = len(valid_embs)
num_splits = len(splits)

# Launch all possible jobs
cv_futures = [
    [ None for _ in range(num_ckpt) ]
    for _ in range(num_splits)
]
test_futures = [ 
    [ None for _ in range(num_ckpt) ]
    for _ in range(num_splits)
]
for split_idx, split in enumerate(splits):
    for ckpt_idx in range(num_ckpt):
        X = valid_embs[ckpt_idx]
        y = valid_labels
        test_mask = to_mask(valid_ids, split["test"])
        train_mask = to_mask(valid_ids, split["train"])
        val_mask = to_mask(valid_ids, split["val"])
        _future = ray_logreg_scores.remote(X, y, train_mask, val_mask)
        cv_futures[split_idx][ckpt_idx] = _future
        
        _future = ray_logreg_scores.remote(X, y, train_mask | val_mask, test_mask)
        test_futures[split_idx][ckpt_idx] = _future
    
test_results_list = []
best_ckpt_idx_list = []
for split_idx, split in tqdm(enumerate(splits), total=len(splits)):
    scores_list = []
    for ckpt_idx in range(num_ckpt):
        results = ray.get(cv_futures[split_idx][ckpt_idx])
        pred, true = results["pred"], results["true"]
        scores = {
            "bacc": balanced_accuracy_score(true, pred),
            "f1": f1_score(true, pred, average='macro'),
            "index": ckpt_idx,
        }
        scores_list.append(scores)

    # Find results on best epoch
    best_scores = sorted(scores_list, key=lambda x: x["f1"], reverse=True)[0]
    best_ckpt_idx = best_scores["index"]
    #print(f"Best index: {best_ckpt_idx}")
    best_ckpt_idx_list.append(best_ckpt_idx)

    test_results = ray.get(test_futures[split_idx][best_ckpt_idx])
    test_results["index"] = best_ckpt_idx
    test_results_list.append(test_results)

pred = np.concatenate([result["pred"] for result in test_results_list])
true = np.concatenate([result["true"] for result in test_results_list])
bacc = balanced_accuracy_score(true, pred)
f1 = f1_score(true, pred, average="macro")
print(f"bacc: {bacc:.4f}")
print(f"f1: {f1:.4f}")
print(best_ckpt_idx_list)

fig = plt.figure(figsize=(3, 3))
ax = fig.add_subplot(111)
cm_props = dict(normalize="true", cmap="Blues", ax=ax, colorbar=False, im_kw={"vmin": 0, "vmax": 1}, text_kw={"fontsize": 8}, xticks_rotation=45)
disp = ConfusionMatrixDisplay.from_predictions(true, pred, **cm_props)
disp.figure_.colorbar(disp.im_, ax=disp.ax_, fraction=0.046, pad=0.04)
disp.figure_.tight_layout()

## Inductive zero-shot

In [ ]:
# Load split

splits_file = Path("../splits/steinmetz_split1.pkl")

with open(splits_file, "rb") as f:
    splits_data = pickle.load(f)

label_map = splits_data["label_map"]
split = splits_data["splits"]

In [ ]:
# Remove UUIDs not known to splits

def to_mask(check_ids, base_ids):
    """
    Returns a bool array indicating if elements of `check_ids` are 
    present in `base_ids`. 
    Output shape: (len(check_ids),)
    """
    base_ids_set = set(base_ids)
    return np.array([x in base_ids_set for x in check_ids])

# Subset units based on what labels are available
valid_id_mask = to_mask(ids, label_map.index.values)
print(f"Valid UUID stats: {valid_id_mask.mean()=}, {(~valid_id_mask).sum()=}")

valid_ids = ids[valid_id_mask]
valid_labels = label_map.loc[valid_ids].to_numpy().ravel()
valid_embs = []
for emb in tqdm(embs):
    valid_embs.append(emb[valid_id_mask])

In [ ]:
#from cuml.linear_model import LogisticRegression
#from cuml.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import ray

seed = 42

def balanced_resampling(X, y):
    resample = RandomOverSampler(random_state=seed)
    resample_idx, _ = resample.fit_resample(np.arange(len(y)).reshape(-1, 1), y)
    resample_idx = resample_idx.ravel()
    return X[resample_idx], y[resample_idx]

def logreg_scores(X, y, train_mask, val_mask):
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    #X_train, y_train = balanced_resampling(X_train, y_train)

    scaler = StandardScaler()
    clf = LogisticRegression(
            max_iter=1000,
            tol=1e-4,
            class_weight="balanced",
            C=1.0,
            solver="newton-cg",
            verbose=0,
    )
    clf.fit(scaler.fit_transform(X_train), y_train)
    pred_val = clf.predict(scaler.transform(X_val))

    return {
        "pred": pred_val,
        "true": y_val,
    }

@ray.remote
def ray_logreg_scores(X, y, train_mask, val_mask):
    return logreg_scores(X, y, train_mask, val_mask)

ray.init(address="local", num_cpus=32, num_gpus=1, ignore_reinit_error=True)

num_ckpt = len(valid_embs)
num_cv_splits = len(split["train_cv"])

# Launch all possible jobs
cv_futures = [
    [ None for _ in range(num_cv_splits) ]
    for _ in range(num_ckpt)
]
test_futures = [None for _ in range(num_ckpt)]
test_mask = to_mask(valid_ids, split["test"])
train_mask = ~test_mask

# Do CV to find best epoch
for ckpt_idx in range(num_ckpt):
    X = valid_embs[ckpt_idx]
    y = valid_labels
    for cv_idx, cv_split in enumerate(split["train_cv"]):
        cv_train_mask = to_mask(valid_ids, cv_split["train"])
        cv_val_mask = to_mask(valid_ids, cv_split["val"])
        _future = ray_logreg_scores.remote(X, y, cv_train_mask, cv_val_mask)
        cv_futures[ckpt_idx][cv_idx] = _future

    _future = ray_logreg_scores.remote(X, y, train_mask, test_mask)
    test_futures[ckpt_idx] = _future

test_results_list = []
scores_list = []
for ckpt_idx in range(num_ckpt):
    results_list = ray.get(cv_futures[ckpt_idx])
    pred = np.concatenate([result["pred"] for result in results_list])
    true = np.concatenate([result["true"] for result in results_list])
    
    test_results = ray.get(test_futures[ckpt_idx])
    test_results_list.append(test_results)
    scores = {
        "val_bacc": balanced_accuracy_score(true, pred),
        "val_f1": f1_score(true, pred, average='macro'),
        "test_bacc": balanced_accuracy_score(test_results["true"], test_results["pred"]),
        "test_f1": f1_score(test_results["true"], test_results["pred"], average='macro'),
        "index": ckpt_idx,
    }
    print(f"CKPT {ckpt_idx} | Val - bacc: {scores['val_bacc']}, f1: {scores['val_f1']} | Test - bacc: {scores['test_bacc']}, f1: {scores['test_f1']}")
    scores_list.append(scores)

# Find results on best epoch
best_scores = sorted(scores_list, key=lambda x: x["val_f1"], reverse=True)[0]
best_ckpt_idx = best_scores["index"]

test_results = test_results_list[best_ckpt_idx]
test_results["index"] = best_ckpt_idx

pred = test_results["pred"]
true = test_results["true"]
bacc = balanced_accuracy_score(true, pred)
f1 = f1_score(true, pred, average="macro")
print(f"bacc: {bacc:.4f}")
print(f"f1: {f1:.4f}")

fig = plt.figure(figsize=(3, 3))
ax = fig.add_subplot(111)
cm_props = dict(normalize="true", cmap="Blues", ax=ax, colorbar=False, im_kw={"vmin": 0, "vmax": 1}, text_kw={"fontsize": 8}, xticks_rotation=45)
disp = ConfusionMatrixDisplay.from_predictions(true, pred, **cm_props)
disp.figure_.colorbar(disp.im_, ax=disp.ax_, fraction=0.046, pad=0.04)
disp.figure_.tight_layout()